# 🏭 CoughTB — Pre-compute Mel Spectrograms

Convert all WAV files to Mel spectrograms **before training** to speed up the training loop by ~10×.

**Why?**
- `librosa.load()` + silence trimming + STFT → mel is expensive
- With many audio files, doing this on-the-fly during training creates a CPU bottleneck
- Pre-computing once saves `.npy` files → training just loads arrays from disk (minimal CPU work)

**Output**: `precomputed_mels/` — each file is `(3, 224, 224)` float32 numpy array, normalized to [0, 1]

## 1. Install Dependencies

In [ ]:
!pip install -q librosa soundfile numpy pandas scipy tqdm

## 2. Set Output Directory

Results will be saved to **local Colab storage** by default.
For larger datasets, you can optionally use Google Drive (Cell 2b).

In [ ]:
import os, shutil
from pathlib import Path

# Default: save to local Colab storage
# Change this to '/content/drive/MyDrive/coughtb_precomputed' if using Drive
OUTPUT_ROOT = '/content/precomputed_mels'
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print(f'Output dir: {OUTPUT_ROOT}')

# Show free disk space
usage = shutil.disk_usage('/content')
print(f'Free space: {usage.free / 1024**3:.1f} GB')
print()
print('⚠️  Colab local storage is TEMPORARY — files will be deleted when runtime ends.')
print('   Download the results before closing Colab, or use Google Drive (Cell 2b).')

## 2b. (Alternative) Use Google Drive instead

**Skip Cell 2 above** and run this cell instead if you want to save to Google Drive.
This keeps your pre-computed mels persistent between Colab sessions.

In [ ]:
# Run THIS CELL instead of Cell 2 if you want to use Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
#
# OUTPUT_ROOT = '/content/drive/MyDrive/coughtb_precomputed'
# os.makedirs(OUTPUT_ROOT, exist_ok=True)
# print(f'Output dir: {OUTPUT_ROOT}')

print('This cell is disabled by default. Uncomment the lines above to use Drive.')

## 3. Download & Extract Dataset

Two options:
1. **Recommended**: Download CODA TB dataset directly from HuggingFace (2.16 GB)
2. **Manual**: Upload your own `dataset.zip` if you have it locally

In [ ]:
import zipfile
import requests
from pathlib import Path

DATA_DIR = Path('/content/dataset')

# ========================
# OPTION 1: Download from HuggingFace (Recommended)
# ========================
DOWNLOAD_FROM_HF = True   # Set to False to upload manually instead

if DOWNLOAD_FROM_HF:
    url = 'https://huggingface.co/datasets/AHFIDAILabs/coda_tb_dataset/resolve/main/dataset.zip'
    zip_path = '/content/dataset.zip'

    print('Downloading CODA TB dataset from HuggingFace...')
    print(f'  URL: {url}')
    print(f'  Size: 2.16 GB')
    print('  This may take 5-15 minutes...')
    print()

    # Stream download with progress
    # ⚠️ If interrupted mid-way, just re-run — the partial file is overwritten safely ('wb' mode)
    response = requests.get(url, stream=True)
    response.raise_for_status()
    total_size = int(response.headers.get('content-length', 0))
    downloaded = 0

    with open(zip_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                pct = downloaded / total_size * 100
                mb_dl = downloaded / 1024 / 1024
                mb_total = total_size / 1024 / 1024
                print(f'\r  Downloading: {mb_dl:.0f}/{mb_total:.0f} MB ({pct:.0f}%)', end='')

    print(f'\n✅ Download complete: {zip_path}')

else:
    # ========================
    # OPTION 2: Upload manually
    # ========================
    from google.colab import files
    print('=' * 60)
    print('Please upload your dataset ZIP file')
    print('=' * 60)
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]
    print(f'Uploaded: {zip_path}')

# ========================
# Extract ZIP
# ========================
print(f'\nExtracting {zip_path}...')
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(str(DATA_DIR))
print(f'✅ Extracted to {DATA_DIR}')

# Find solicited_data
solicited_dirs = list(DATA_DIR.rglob('solicited_data'))
if solicited_dirs:
    solicited_dir = solicited_dirs[0]
    wav_files = sorted(solicited_dir.glob('*.wav'))
    print(f'\n✅ Solicited data dir: {solicited_dir}')
    print(f'   Total WAV files: {len(wav_files)}')
else:
    print('❌ Could not find solicited_data directory')
    print('Available structure:')
    for f in list(DATA_DIR.rglob('*'))[:50]:
        print(f'  {f.relative_to(DATA_DIR)}')
    raise FileNotFoundError('Dataset not found - check the download')

## 4. Load & Explore Metadata

We need `tb_status` (label) for each cough file. This comes from merging the solicited metadata with clinical metadata.

In [ ]:
import pandas as pd

# Find metadata CSV files
clinical_paths = list(DATA_DIR.rglob('*Clinical_Meta_Info.csv'))
solicited_meta_paths = list(DATA_DIR.rglob('*Solicited_Meta_Info.csv'))

print(f'Clinical CSVs: {[p.relative_to(DATA_DIR) for p in clinical_paths]}')
print(f'Solicited CSVs: {[p.relative_to(DATA_DIR) for p in solicited_meta_paths]}')

clinical = pd.read_csv(clinical_paths[0])
solicited_meta = pd.read_csv(solicited_meta_paths[0])

print(f'\nClinical: {clinical.shape}')
print(f'  Columns: {list(clinical.columns)}')
print(f'  tb_status distribution: {clinical["tb_status"].value_counts().to_dict()}')

print(f'\nSolicited Meta: {solicited_meta.shape}')
print(f'  Columns: {list(solicited_meta.columns)}')
print(f'  Unique participants: {solicited_meta["participant"].nunique()}')

In [ ]:
# Merge to get tb_status for each cough filename
metadata = solicited_meta.merge(clinical[['participant', 'tb_status']], on='participant', how='left')

# Check which WAV files actually exist
existing_filenames = set(f.name for f in wav_files)
metadata['wav_exists'] = metadata['filename'].isin(existing_filenames)
available = metadata[metadata['wav_exists']].copy()

print(f'Metadata entries: {len(metadata)}')
print(f'Existing WAV files: {len(existing_filenames)}')
print(f'Matched (available): {len(available)}')
print(f'  TB+: {(available["tb_status"]==1).sum()}')
print(f'  TB-: {(available["tb_status"]==0).sum()}')
print(f'  Missing tb_status: {available["tb_status"].isna().sum()}')

## 5. Define Preprocessing Functions

These are **identical** to the functions used in the training notebook — so the pre-computed data will be byte-for-byte compatible.

In [ ]:
import numpy as np
import librosa
from scipy.ndimage import zoom

SR = 16000
CROP = 0.5  # seconds
N_MELS = 224
FMAX = 8000
TARGET_LEN = int(SR * CROP)  # 8000 samples

def load_and_segment(path):
    """Load audio, trim silence, extract 0.5s with highest energy."""
    y, _ = librosa.load(path, sr=SR)
    if len(y) == 0:
        return None
    y, _ = librosa.effects.trim(y, top_db=20)
    if len(y) >= TARGET_LEN:
        energy = np.convolve(y**2, np.ones(TARGET_LEN), mode='valid')
        start = np.argmax(energy)
        seg = y[start:start+TARGET_LEN]
    else:
        seg = np.zeros(TARGET_LEN, dtype=y.dtype)
        seg[:len(y)] = y
    return seg


def make_mel_rgb(y_seg):
    """Convert audio segment to mel spectrogram (3, 224, 224), normalized to [0, 1].
    No augmentation applied — augmentation happens during training."""
    mel = librosa.feature.melspectrogram(
        y=y_seg, sr=SR, n_mels=N_MELS, fmax=FMAX,
        hop_length=512, win_length=2048
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)

    target_shape = (224, 224)
    if mel_db.shape != target_shape:
        zf = (target_shape[0] / mel_db.shape[0], target_shape[1] / mel_db.shape[1])
        resized = zoom(mel_db, zf, order=1)
    else:
        resized = mel_db

    if np.ptp(resized) > 0:
        normed = (resized - resized.min()) / np.ptp(resized)
    else:
        normed = np.zeros_like(resized)

    rgb = np.stack([normed] * 3, axis=0).astype(np.float32)
    return rgb


print('✅ Preprocessing functions defined')
print(f'  Sample rate: {SR} Hz')
print(f'  Segment: {CROP}s ({TARGET_LEN} samples)')
print(f'  Output: (3, 224, 224) float32')
print(f'  Memory per file: {3 * 224 * 224 * 4 / 1024:.1f} KB')

## 6. Pre-compute All Mel Spectrograms

This cell processes all WAV files and saves `.npy` files.

**Expected time**: ~5-15 minutes for 10k files, ~30-60 minutes for 55k files
**Storage**: ~450 MB for 55k files

In [ ]:
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

# Output directory for .npy files
MELS_DIR = Path(OUTPUT_ROOT) / 'precomputed_mels'
MELS_DIR.mkdir(parents=True, exist_ok=True)

# Count already done
existing_npy = set(f.stem for f in MELS_DIR.glob('*.npy'))
print(f'Already pre-computed: {len(existing_npy)} files')

errors = []
skipped = 0

# Only process files that match available metadata
to_process = available[~available['filename'].str.replace('.wav', '', case=False).isin(existing_npy)]
print(f'To process this session: {len(to_process)} files')
print(f'\nStarting pre-computation...')

for idx in tqdm(range(len(to_process)), desc='Pre-computing mels'):
    row = to_process.iloc[idx]
    wav_name = row['filename']
    stem = Path(wav_name).stem
    npy_path = MELS_DIR / f'{stem}.npy'

    if npy_path.exists():
        skipped += 1
        continue

    try:
        y_seg = load_and_segment(str(solicited_dir / wav_name))
        if y_seg is None:
            errors.append((wav_name, 'empty after load_and_segment'))
            continue
        mel = make_mel_rgb(y_seg)
        np.save(str(npy_path), mel)
    except Exception as e:
        errors.append((wav_name, str(e)))

    # Small progress indicator
    if (idx + 1) % 25000 == 0:
        print(f'  [{idx+1}/{len(to_process)}] Saved to {MELS_DIR}')

print(f'\n✅ Pre-computation complete!')
print(f'   Total .npy files: {len(list(MELS_DIR.glob("*.npy")))}')
print(f'   Skipped (already done): {skipped}')
print(f'   Errors: {len(errors)}')
if errors:
    print(f'   First 5 errors:')
    for wav, err in errors[:5]:
        print(f'     - {wav}: {err}')

## 7. Create Metadata CSV for Training

Save a CSV that maps each `.npy` file path to its `tb_status` label and `participant` ID.
The training notebook will load this CSV instead of re-scanning WAV files.

In [ ]:
# Build the mapping for all successfully pre-computed files
completed_npy = set(f.stem for f in MELS_DIR.glob('*.npy'))

mapping = available[available['filename'].str.replace('.wav', '', case=False).isin(completed_npy)].copy()
mapping['mel_path'] = mapping['filename'].apply(
    lambda x: str(MELS_DIR / f'{Path(x).stem}.npy')
)

print(f'Total pre-computed files: {len(mapping)}')
print(f'  TB+: {(mapping["tb_status"]==1).sum()}')
print(f'  TB-: {(mapping["tb_status"]==0).sum()}')
print(f'  Unique participants: {mapping["participant"].nunique()}')

# Save the mapping CSV
csv_path = Path(OUTPUT_ROOT) / 'precomputed_metadata.csv'
mapping.to_csv(csv_path, index=False)
print(f'\n✅ Metadata saved: {csv_path}')
print(f'   Columns: {list(mapping.columns)}')
print(f'   Size: {csv_path.stat().st_size / 1024:.1f} KB')

# Also save a small sample for testing
sample = mapping.groupby('tb_status').sample(n=50, random_state=42, replace=True)
sample_path = Path(OUTPUT_ROOT) / 'precomputed_sample.csv'
sample.to_csv(sample_path, index=False)
print(f'\n✅ Sample (up to 100 files) saved: {sample_path}')

# Verify a sample
print('\nVerifying sample files...')
sample_ok = sum(1 for p in sample['mel_path'].values if os.path.exists(p))
print(f'  {sample_ok}/{len(sample)} .npy files exist')

## 8. Download Pre-computed Data

Since Colab local storage is temporary, download the results now.
You need:
1. `precomputed_metadata.csv` — maps filenames → labels
2. `.npy` files in `precomputed_mels/` — the pre-computed mel spectrograms

**If using Google Drive**: Files are already persistent — just skip this.

In [ ]:
from google.colab import files
import shutil

print('=== Download Options ===')
print()

# Option 1: Download just the metadata CSV
print('Option 1: Download metadata CSV only...')
files.download(str(csv_path))
print()

# Option 2: Create + download a test subset (1000 files)
print('Option 2: Creating test subset (1000 files)...')
TEST_SUBSET_SIZE = 1000
test_subset = mapping.head(TEST_SUBSET_SIZE).copy()
test_subset_csv = Path(OUTPUT_ROOT) / 'precomputed_test_subset.csv'
test_subset.to_csv(test_subset_csv, index=False)

test_mels_dir = Path(OUTPUT_ROOT) / 'precomputed_mels_test_subset'
test_mels_dir.mkdir(exist_ok=True)
for _, row in tqdm(test_subset.iterrows(), desc='Copying test subset', total=len(test_subset)):
    src = row['mel_path']
    dst = test_mels_dir / Path(src).name
    if os.path.exists(src) and not dst.exists():
        shutil.copy2(src, dst)

zip_path = '/content/precomputed_test_subset.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(str(test_subset_csv), 'precomputed_metadata.csv')
    for npy_file in test_mels_dir.glob('*.npy'):
        zf.write(str(npy_file), f'precomputed_mels/{npy_file.name}')

print(f'\n✅ Test subset ZIP created')
print(f'   Files: {len(test_subset)} mels + 1 CSV')
print(f'   Size: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB')
files.download(zip_path)

---
## Summary

### What was saved

| Item | Location | Description |
|:-----|:---------|:------------|
| Mel `.npy` files | `precomputed_mels/` | `(3, 224, 224)` float32 arrays |
| Metadata CSV | `precomputed_metadata.csv` | Maps filename to label |
| Test subset | `precomputed_test_subset/` | 1000 files for quick testing |

### Next step
1. Download `precomputed_test_subset.zip` (or full pre-computed data)
2. Open `training.ipynb` in Colab
3. Upload the pre-computed ZIP → train!